In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent / "src"))

from lcoe import lcoe_heat

import geopandas as gpd

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [2]:
# Input and output paths
ROOT = Path().resolve().parent
RAW  = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed' 

In [3]:
raw_data = pd.read_csv(PROCESSED / "dataset_filtered.csv")
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2525 entries, 0 to 2524
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   index                  2525 non-null   int64  
 1   well_id                2525 non-null   str    
 2   easting                2525 non-null   float64
 3   northing               2525 non-null   float64
 4   depth_tvd_m            2525 non-null   float64
 5   porosity_pct           2525 non-null   float64
 6   bulk_density_gcc       2525 non-null   float64
 7   formation_thickness_m  2525 non-null   float64
 8   distance_to_usp_km     2525 non-null   float64
 9   temp_harrison          2525 non-null   float64
 10  temp_res_c             2525 non-null   float64
 11  permeability_md        2525 non-null   float64
 12  k_log                  2525 non-null   float64
dtypes: float64(11), int64(1), str(1)
memory usage: 256.6 KB


In [4]:
df = raw_data.copy()
well_list = df["well_id"].unique().tolist()

In [5]:
HEATING_TARGET = 10 #MW
COOLING_TARGET = 5 #MW

MAX_LEGAL_PRESSURE_PA = 50e5  # PLACEHOLDER -- replace with actual SodM/regulatory limit once found

In [6]:
wells_base = {
    row["well_id"]: {
        "Thickness": row["formation_thickness_m"],
        "Permeability": row["permeability_md"] * 9.869e-16,
        "Transmissivity": row["formation_thickness_m"] * row["permeability_md"] * 9.869e-16,
        "Temperature": row["temp_res_c"],
        "Depth": row["depth_tvd_m"]
    }
    for _, row in df.iterrows()
}

### Flow Rate and Thermal Power

$$\mu = 2.414 \times 10^{-5} \times 10^{\frac{247.8}{T + 133.15}}$$

$$Q=\frac{2\pi T\Delta P}{\mu (\ln (r_{e}/r_{w}) + S)}$$

$$ P_{th} = \dot{m}C_p(T_p - T_r)$$

In [7]:
def flow_rate_and_thermal_power(kh, T, T_r=40, rho_water=970, cp_water=4186, delta_p=18e5, r_e=1000, r_w=8 * 0.0254 / 2, skin=0):
    """
    Calculates radial flow rate (q) and thermal power (P_th) for a water well.
    
    Parameters:
    kh        : Permeability-thickness product in m^3 (Transmissibility)
    T         : Temperature of the produced water in Celsius
    T_r       : Reinjection/reference temperature in Celsius (default: 40 C)
    rho_water : Density of water in kg/m^3 (default: 970)
    cp_water  : Specific heat capacity of water in J/(kg*K) (default: 4186)
    delta_p   : Pressure drawdown in Pascals (default: 1.8 MPa)
    r_e       : External/drainage radius in meters (default: 1000 m)
    r_w       : Wellbore radius in meters (default: 4 inches converted to meters)
    
    Returns:
    q         : Volumetric flow rate in m^3/s
    P_th_MW   : Thermal power in Megawatts (MW)
    """
    # Calculate dynamic viscosity of water as a function of temperature (Celsius)
    mu = 2.414e-5 * 10 ** (247.8 / (T + 133.15))

    # Radial flow rate using Darcy's Law
    q = (2 * np.pi * kh * delta_p) / (mu * (np.log(r_e / r_w) + skin))

    # Mass flow rate (kg/s)
    m_dot = rho_water * q
    T_p = T 

    # Thermal power in Watts, then converted to Megawatts
    P_th = m_dot * cp_water * (T_p - T_r)
    P_th_MW = P_th / 1e6

    return q, P_th_MW

In [8]:
def evaluate(well, power):
    gap = HEATING_TARGET - power
    if power > HEATING_TARGET:
        print(f"\n{well}meets both heating and cooling targets.\nTarget exceeded by {-gap:.2f} MW")
    elif power > COOLING_TARGET:
        print(f"\n{well} meets cooling target only.\n{gap:.2f} MW required to reach heating target.")
    else: 
        print(f"\n{well} does not meet heating or cooling target.\n{gap:.2f} MW required to reach heating target.")

    print(f"{well}'s thermal power is {power:.2f} MW")

## Production Scenarios

### 1. One Doublet

In [9]:
q_blt_base, p_blt_base = flow_rate_and_thermal_power(kh=wells_base["BLT-01"]["Transmissivity"], T=wells_base["BLT-01"]["Temperature"])
q_jut_base, p_jut_base = flow_rate_and_thermal_power(kh=wells_base["JUT-01"]["Transmissivity"], T=wells_base["JUT-01"]["Temperature"])

print("===BaseCase Power===")
print(f"Heating Target = {HEATING_TARGET} MW")
print(f"Cooling Target = {COOLING_TARGET} MW")
evaluate("BLT-01", p_blt_base)
evaluate("JUT-01", p_jut_base)

===BaseCase Power===
Heating Target = 10 MW
Cooling Target = 5 MW

BLT-01 meets cooling target only.
4.43 MW required to reach heating target.
BLT-01's thermal power is 5.57 MW

JUT-01 does not meet heating or cooling target.
8.03 MW required to reach heating target.
JUT-01's thermal power is 1.97 MW


### 2. Two Doublets

In [10]:
p_total_base = p_blt_base + p_jut_base

print("===BaseCase Power===")
print(f"Heating Target = {HEATING_TARGET} MW")
print(f"Cooling Target = {COOLING_TARGET} MW")
evaluate("BLT-01 and JUT-01", p_total_base)

===BaseCase Power===
Heating Target = 10 MW
Cooling Target = 5 MW

BLT-01 and JUT-01 meets cooling target only.
2.46 MW required to reach heating target.
BLT-01 and JUT-01's thermal power is 7.54 MW


In [11]:
increase = (HEATING_TARGET - p_total_base) / p_total_base * 100

print("In order to meet our heating target, we need an increase in thermal power.\n"
    f"An increase in thermal power by {increase:.1f}% can be achieved by boosting flow rate by {increase:.1f}%\n"
    "(via stimulation, artificial lift/ESPs, or lateral sidetracks) "
    "or by lowering the reinjection temperature (T_r).")

In order to meet our heating target, we need an increase in thermal power.
An increase in thermal power by 32.7% can be achieved by boosting flow rate by 32.7%
(via stimulation, artificial lift/ESPs, or lateral sidetracks) or by lowering the reinjection temperature (T_r).


## Economics: Well Interventions and LCOE

In [12]:
# Stimulation options. 'none' uses TNO ThermoGIS base skin (-1, not 0) for a
# deviated (45deg) well trajectory. 'thermogis_stim' is TNO's own referenced
# combined-well stimulation scenario (skin -4, EUR 0.5M for BOTH wells).
interventions = {
    "none":           dict(skin=-1, cost_mln=0.0),
    "thermogis_stim": dict(skin=-4, cost_mln=0.5 / 2),
}

In [13]:
def compute_well_power(well, method):
    params = interventions[method]
    q, p = flow_rate_and_thermal_power(
        kh=wells_base[well]["Transmissivity"],
        T=wells_base[well]["Temperature"],
        skin=params["skin"]
    )
    return p

In [14]:
def compute_well_lcoe(well, power_mw, stim_cost_mln=0.0):
    return lcoe_heat(
        heat_mwth=power_mw,
        depth_m=wells_base[well]["Depth"],
        n_wells=2,
        stim_cost_per_well_mln=stim_cost_mln,
    )["LCOE_eur_per_GJ"]

### Grid Search

In [15]:
import itertools
from itertools import chain, combinations

def well_subsets(wells):
    return chain.from_iterable(combinations(wells, r) for r in range(1, len(wells) + 1))

results = []
for subset in well_subsets(well_list):
    for combo in itertools.product(interventions.keys(), repeat=len(subset)):
        detail = {}
        total_power = 0
        lcoe_values = []

        for well, method in zip(subset, combo):
            p = compute_well_power(well, method)
            lcoe_gj = compute_well_lcoe(well, p, stim_cost_mln=interventions[method]["cost_mln"])
            detail[well] = method
            detail[f"{well}_MW"] = round(p, 2)
            detail[f"{well}_LCOE_eur_GJ"] = round(lcoe_gj, 2)
            total_power += p
            lcoe_values.append(lcoe_gj)

        for well in well_list:
            if well not in subset:
                detail[well] = "not_drilled"
                detail[f"{well}_MW"] = 0.0
                detail[f"{well}_LCOE_eur_GJ"] = None

        detail["n_wells_used"] = len(subset)
        detail["total_MW"] = round(total_power, 2)
        detail["avg_LCOE_eur_GJ"] = round(sum(lcoe_values) / len(lcoe_values), 2)
        detail["meets_heating"] = total_power >= HEATING_TARGET
        results.append(detail)

df_grid = pd.DataFrame(results).sort_values("avg_LCOE_eur_GJ")
df_grid


,BLT-01,BLT-01_MW,BLT-01_LCOE_eur_GJ,JUT-01,JUT-01_MW,JUT-01_LCOE_eur_GJ,n_wells_used,total_MW,avg_LCOE_eur_GJ,meets_heating
1,thermogis_stim,9.85,7.71,not_drilled,0.00,NaN,1,9.85,7.71,False
0,none,6.25,10.30,not_drilled,0.00,NaN,1,6.25,10.30,False
7,thermogis_stim,9.85,7.71,thermogis_stim,3.49,15.81,2,13.34,11.76,True
5,none,6.25,10.30,thermogis_stim,3.49,15.81,2,9.73,13.06,False
6,thermogis_stim,9.85,7.71,none,2.21,22.32,2,12.06,15.02,True
3,not_drilled,0.00,NaN,thermogis_stim,3.49,15.81,1,3.49,15.81,False
4,none,6.25,10.30,none,2.21,22.32,2,8.46,16.31,False
2,not_drilled,0.00,NaN,none,2.21,22.32,1,2.21,22.32,False


## Monte-Carlo